# Introduction to NetCDF Files

This notebook introduces the **NetCDF** (Network Common Data Form) file format, which is widely used in atmospheric and environmental science for storing self-describing, array-oriented scientific data.

By the end of this notebook you will be able to:
- Open a NetCDF file and inspect its structure
- Read global metadata (attributes)
- List dimensions and variables
- Read variable data and attributes (units, QC flags, etc.)
- Load the data into a pandas DataFrame for analysis
- Make a simple plot

The example data comes from the **NCAS Chilbolton Atmospheric Observatory** (CAO) wind measurements.

## 1. What is NetCDF?

NetCDF is a binary file format designed for storing multidimensional scientific array data. Key properties:

| Feature | Description |
|---|---|
| **Self-describing** | The file contains metadata (attributes) that describe what the data means |
| **Portable** | Binary format is the same on all platforms |
| **Scalable** | Handles data from kilobytes to terabytes |
| **CF Conventions** | Most atmospheric science NetCDF files follow the [Climate and Forecast (CF) conventions](https://cfconventions.org/) for variable names and units |

A NetCDF file is organised into:
- **Dimensions** — named axes (e.g. `time`, `latitude`, `longitude`)
- **Variables** — arrays indexed by one or more dimensions
- **Attributes** — metadata attached to the file (global) or to individual variables

## 2. Opening a NetCDF File

We use the `netCDF4` Python library. Always open files with a `with` block so they are closed automatically.

In [ ]:
import netCDF4 as nc4
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Path to a single example file
EXAMPLE_FILE = '/data/wexp/cwalden/mean-winds/2014/ncas-anemometer-2_cao_20140401_mean-winds_v1.1.nc'

with nc4.Dataset(EXAMPLE_FILE) as nc:
    print(type(nc))
    print(nc)

## 3. Global Attributes

Global attributes describe the dataset as a whole — who created it, what instrument was used, what time period it covers, etc.

In [ ]:
with nc4.Dataset(EXAMPLE_FILE) as nc:
    print('=== Global Attributes ===')
    for attr in nc.ncattrs():
        print(f'  {attr}: {getattr(nc, attr)}')

Notice useful fields like:
- `title` — human-readable description
- `time_coverage_start` / `time_coverage_end` — date range
- `platform_name` — location of the instrument
- `sampling_interval` / `averaging_interval` — temporal resolution
- `Conventions` — which metadata standard is followed (here CF-1.6 and NCAS-GENERAL)

## 4. Dimensions and Variables

Dimensions define the shape of the arrays. Variables hold the actual data.

In [ ]:
with nc4.Dataset(EXAMPLE_FILE) as nc:
    print('=== Dimensions ===')
    for name, dim in nc.dimensions.items():
        print(f'  {name}: {len(dim)}')

    print()
    print('=== Variables ===')
    for name, var in nc.variables.items():
        print(f'  {name}:  shape={var.shape}  dtype={var.dtype}  dims={var.dimensions}')

## 5. Variable Attributes

Each variable has its own attributes such as `units`, `long_name`, `_FillValue`, and QC flag meanings.

In [ ]:
with nc4.Dataset(EXAMPLE_FILE) as nc:
    for varname in ['wind_speed', 'wind_from_direction', 'qc_flag_wind_speed', 'time']:
        var = nc.variables[varname]
        print(f'--- {varname} ---')
        for attr in var.ncattrs():
            print(f'  {attr}: {getattr(var, attr)}')
        print()

Key things to note:
- `units` tells you what the numbers mean (e.g. `m s-1`, `degree`, `seconds since 1970-01-01`)
- `_FillValue` is the placeholder used where data is missing — **do not treat this as a real measurement**
- `flag_values` and `flag_meanings` describe the QC flags:
  - `0` = not used
  - `1` = good data
  - `2` = bad data

## 6. Reading Data into NumPy Arrays

Variables behave like NumPy arrays. Use `.data.copy()` to get a plain array without a netCDF4 mask.

In [ ]:
with nc4.Dataset(EXAMPLE_FILE) as nc:
    unix_time  = nc.variables['time'][:].data.copy().astype(np.float64)
    wind_speed = nc.variables['wind_speed'][:].data.copy().astype(np.float64)
    wind_dir   = nc.variables['wind_from_direction'][:].data.copy().astype(np.float64)
    qc_speed   = nc.variables['qc_flag_wind_speed'][:].data.copy().astype(np.int8)
    qc_dir_var = (nc.variables.get('qc_flag_wind_from_direction')
                  or nc.variables.get('qc_flag_wind_direction'))
    qc_dir     = qc_dir_var[:].data.copy().astype(np.int8) if qc_dir_var is not None else None

print(f'Number of time steps: {len(unix_time)}')
print(f'Wind speed array (first 5 values): {wind_speed[:5]}')
print(f'QC flags for wind speed (first 5): {qc_speed[:5]}')

## 7. Applying QC Flags

Only flag value `1` means *good data*. Flag `0` means *not used*, and flag `2` (or higher) means *bad data*. We set bad values to `NaN` so they are ignored in calculations and plots.

In [ ]:
# Apply QC: set anything that is not flag 0 or 1 to NaN
wind_speed_qc = wind_speed.copy()
wind_speed_qc[(qc_speed != 0) & (qc_speed != 1)] = np.nan

wind_dir_qc = wind_dir.copy()
if qc_dir is not None:
    wind_dir_qc[(qc_dir != 0) & (qc_dir != 1)] = np.nan

n_total = len(wind_speed)
n_good  = np.sum(~np.isnan(wind_speed_qc))
print(f'Total records:  {n_total}')
print(f'Good records:   {n_good}  ({100*n_good/n_total:.1f}%)')
print(f'Flagged/missing: {n_total - n_good}  ({100*(n_total - n_good)/n_total:.1f}%)')

## 8. Converting to a pandas DataFrame

A DataFrame is much easier to work with for time-series analysis. The `time` variable holds **Unix time** (seconds since 1970-01-01 UTC), which pandas can convert directly.

In [ ]:
df = pd.DataFrame({
    'time':      pd.to_datetime(unix_time, unit='s', utc=True),
    'speed':     wind_speed_qc,
    'direction': wind_dir_qc,
})

print(df.head(10))
print()
print(df.describe())

## 9. A Simple Plot

Let's plot the wind speed time series for this single day.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].plot(df['time'], df['speed'], linewidth=0.8)
axes[0].set_ylabel('Wind speed (m s⁻¹)')
axes[0].set_title('Chilbolton wind — 1 April 2014')
axes[0].grid(True, alpha=0.3)

axes[1].plot(df['time'], df['direction'], linewidth=0.8, color='tab:orange')
axes[1].set_ylabel('Wind direction (°)')
axes[1].set_ylim(0, 360)
axes[1].set_yticks([0, 90, 180, 270, 360])
axes[1].set_yticklabels(['N', 'E', 'S', 'W', 'N'])
axes[1].grid(True, alpha=0.3)

axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
axes[1].set_xlabel('Time (UTC)')

fig.tight_layout()
plt.show()

## 10. Reading Multiple Files

A real dataset will span many files — one per day. Use `pathlib.Path.rglob()` to find them all and concatenate the results.

In [ ]:
from pathlib import Path

ROOT = '/data/wexp/cwalden/mean-winds'
files = sorted(Path(ROOT).rglob('*.nc'))
print(f'Found {len(files)} files')
print('First:', files[0].name)
print('Last: ', files[-1].name)

In [ ]:
# Load a full year (2020) as an example
year_files = [f for f in files if '2020' in f.name]
print(f'Files for 2020: {len(year_files)}')

chunks = []
for f in year_files:
    with nc4.Dataset(str(f)) as nc:
        unix  = nc.variables['time'][:].data.copy().astype(np.float64)
        spd   = nc.variables['wind_speed'][:].data.copy().astype(np.float64)
        qc    = nc.variables['qc_flag_wind_speed'][:].data.copy().astype(np.int8)
    spd[(qc != 0) & (qc != 1)] = np.nan
    chunks.append(pd.DataFrame({'unix': unix, 'speed': spd}))

df_2020 = (pd.concat(chunks, ignore_index=True)
             .sort_values('unix')
             .reset_index(drop=True))
df_2020['time'] = pd.to_datetime(df_2020['unix'], unit='s', utc=True)
df_2020 = df_2020.drop(columns='unix')

print(f'Loaded {len(df_2020):,} records')
print(f'Date range: {df_2020["time"].min().date()} to {df_2020["time"].max().date()}')
print(f'Mean wind speed: {df_2020["speed"].mean():.2f} m/s')

## Summary

| Step | Code pattern |
|---|---|
| Open a file | `with nc4.Dataset(path) as nc:` |
| Global attributes | `nc.ncattrs()`, `getattr(nc, name)` |
| List variables | `nc.variables.keys()` |
| Read a variable | `nc.variables['name'][:].data.copy()` |
| Variable attributes | `var.ncattrs()`, `getattr(var, name)` |
| Apply QC flags | Set values where `qc != 1` to `np.nan` |
| Convert time | `pd.to_datetime(unix_array, unit='s', utc=True)` |

You are now ready to work with the full Chilbolton dataset in the analysis notebooks.